In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt 
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import re
import string
import requests
from bs4 import BeautifulSoup

In [ ]:
# Load datasets correctly
data_fake = pd.read_csv("Fake.csv")
data_true = pd.read_csv("True.csv")
print("Loaded Fake:", data_fake.shape, "Loaded True:", data_true.shape)


In [ ]:
# Clean publisher signatures from True news to prevent structural shortcut learning
data_true["text"] = data_true["text"].apply(lambda x: re.sub(r'^.*?\(\s*Reuters\s*\)\s*-\s*', '', str(x)))
data_fake["text"] = data_fake["text"].apply(lambda x: str(x))

# Combine Title and Text to capture headline clickbait and style signals
data_fake["text"] = data_fake["title"].fillna('') + " " + data_fake["text"].fillna('')
data_true["text"] = data_true["title"].fillna('') + " " + data_true["text"].fillna('')

# Assign binary target labels (0 = Fake, 1 = Real)
data_fake["class"] = 0
data_true["class"] = 1

In [ ]:
data_fake.head()

In [ ]:
data_true.head()

In [ ]:
data_fake["class"]=0
data_true["class"]=1

In [ ]:
data_fake.shape, data_true.shape

In [ ]:
data_fake_manual_testing = data_fake.tail(10).copy()
data_true_manual_testing = data_true.tail(10).copy()

data_fake = data_fake.iloc[:-10].copy()
data_true = data_true.iloc[:-10].copy()

In [ ]:
data_fake.shape, data_true.shape

In [ ]:
data_fake_manual_testing['class']=0
data_true_manual_testing['class']=1

In [ ]:
data_fake_manual_testing.head(10)

In [ ]:
data_true_manual_testing.head(10)

In [ ]:
data_merge=pd.concat([data_fake,data_true], axis =0)
data_merge.head(10)

In [ ]:
data_merge.columns

In [ ]:
# Combine headline title with body text for better context before dropping title
data_merge['text'] = data_merge['title'] + " " + data_merge['text']
data = data_merge.drop(['title', 'subject', 'date'], axis=1)

In [ ]:
data.isnull().sum()

In [ ]:
data=data.sample(frac=1)

In [ ]:
data.reset_index(inplace=True)
data.drop(['index'],axis=1,inplace=True)

In [ ]:
data.columns

In [ ]:
data.head()

In [ ]:
def wordopt(text):
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)       # Remove URLs FIRST before stripping punctuation
    text = re.sub(r'read more at:.*', '', text)             # Remove copy-paste link footers (like TOI links)
    text = re.sub(r'^\s*[a-z\s]+:\s*', '', text)            # Remove datelines like "NEW DELHI:" at start
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'<.*?>+', '', text)
    text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub(r'\n', '', text)
    text = re.sub(r'\w*\d\w*', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [ ]:
data['text'] = data['text'].apply(wordopt)

In [ ]:
x= data['text']
y=data['class']

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Expanded TF-IDF vectorizer with sublinear scaling
vectorization = TfidfVectorizer(
    stop_words='english', 
    ngram_range=(1, 2), 
    max_features=50000,
    sublinear_tf=True,
    min_df=2
)

xv_train = vectorization.fit_transform(x_train)
xv_test = vectorization.transform(x_test)

In [ ]:
from sklearn.linear_model import LogisticRegression
LR=LogisticRegression()
LR.fit(xv_train,y_train)

In [ ]:
pred_ir=LR.predict(xv_test)

In [ ]:
LR.score(xv_test,y_test)

In [ ]:
print(classification_report(y_test,pred_ir))

In [ ]:
from sklearn.tree import DecisionTreeClassifier
DT=DecisionTreeClassifier()
DT.fit(xv_train,y_train)

In [ ]:
pred_dt=DT.predict(xv_test)

In [ ]:
DT.score(xv_test,y_test)

In [ ]:
print(classification_report(y_test,pred_dt))

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
GB=GradientBoostingClassifier(random_state=0)
GB.fit(xv_train,y_train)

In [ ]:
predict_gb=GB.predict(xv_test)

In [ ]:
GB.score(xv_test,y_test)

In [ ]:
print(classification_report(y_test,predict_gb))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
RF=RandomForestClassifier(random_state=0)
RF.fit(xv_train,y_train)

In [ ]:
pred_rf=RF.predict(xv_test)

In [ ]:
RF.score(xv_test, y_test)

In [ ]:
from sklearn.ensemble import VotingClassifier

# Build and fit Soft Voting Ensemble
ensemble_model = VotingClassifier(
    estimators=[
        ('lr', LR),
        ('rf', RF),
        ('gb', GB)
    ],
    voting='soft'
)

ensemble_model.fit(xv_train, y_train)
print("Ensemble Accuracy Score:", ensemble_model.score(xv_test, y_test))

In [ ]:
import requests
from bs4 import BeautifulSoup
import re

def fetch_text_from_url(url):
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
        response = requests.get(url, headers=headers, timeout=5)
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Filter out navigation menu and short footer paragraphs (< 30 chars)
        paragraphs = [p.get_text().strip() for p in soup.find_all('p') if len(p.get_text().strip()) > 30]
        article_text = ' '.join(paragraphs)
        
        return article_text if len(article_text.strip()) > 100 else url
    except Exception:
        return url

def output_label(n):
    return "Real News" if n == 1 else "Fake News"

def manual_testing(news):
    raw_input = news.strip()
    
    # Extract URL if present anywhere in input string
    url_match = re.search(r'https?://\S+', raw_input)
    if url_match:
        target_url = url_match.group(0)
        print(f"🌐 URL detected! Scraping content from: {target_url}")
        extracted_text = fetch_text_from_url(target_url)
        if extracted_text == target_url:
            print("⚠️ Scraping failed or blocked. Evaluating raw input text.")
        news_input = extracted_text
    else:
        news_input = raw_input

    # Clean and vectorize input text
    cleaned_text = wordopt(news_input)
    new_xv_test = vectorization.transform([cleaned_text])

    # Vocabulary coverage check
    if new_xv_test.nnz == 0:
        print("⚠️ Warning: Input text contains no words present in the model's vocabulary!")

    # Probability predictions
    prob_LR = LR.predict_proba(new_xv_test)[0][1]
    prob_RF = RF.predict_proba(new_xv_test)[0][1]
    prob_GB = GB.predict_proba(new_xv_test)[0][1]
    prob_ensemble = ensemble_model.predict_proba(new_xv_test)[0][1]

    # Weighted average probability
    avg_prob = (prob_LR + prob_RF + prob_GB + prob_ensemble) / 4.0
    
    # Dynamic threshold for modern out-of-domain articles
    verdict = "Real News" if avg_prob >= 0.45 else "Fake News"
    confidence = avg_prob * 100 if verdict == "Real News" else (1 - avg_prob) * 100

    print("\n==============================")
    print(f" FINAL VERDICT : {verdict.upper()}")
    print(f" CONFIDENCE    : {confidence:.2f}%")
    print("==============================")
    print("Model Predictions Breakout:")
    print("  • Logistic Regression :", output_label(LR.predict(new_xv_test)[0]))
    print("  • Decision Tree       :", output_label(DT.predict(new_xv_test)[0]))
    print("  • Gradient Boosting   :", output_label(GB.predict(new_xv_test)[0]))
    print("  • Random Forest       :", output_label(RF.predict(new_xv_test)[0]))
    print("  • Voting Ensemble     :", output_label(ensemble_model.predict(new_xv_test)[0]))

In [ ]:
news = str(input())
manual_testing(news)

In [ ]:
news = str(input())
manual_testing(news)

In [ ]:
news = str(input())
manual_testing(news)

In [ ]:
news = str(input())
manual_testing(news)